
# Analisis Data Bersih — Skrining Bantul 2025

Notebook ini menganalisis data yang sudah dibersihkan oleh
`Persiapan_Data_Revised.ipynb`. Isinya dua bagian:

- **Bagian A — Apa yang berubah setelah pembersihan.** Membuktikan bahwa data ini layak
  dipakai untuk analisis, bukan sekadar layak untuk model.
- **Bagian B — Siapa populasinya dan siapa yang berisiko.** Profil demografi,
  antropometri, prevalensi risiko per subkelompok, dan komorbiditas.

---

### Tiga catatan yang berlaku untuk SELURUH angka di notebook ini

1. **Semua angka berlaku per catatan skrining, bukan per orang.** Kolom `id` sengaja
   diacak untuk melindungi data asli BPJS, sehingga deduplikasi tingkat individu tidak
   mungkin dilakukan. Bila ada orang yang tercatat lebih dari sekali, ia terhitung lebih
   dari sekali.

2. **Setiap target punya rentang usia sendiri.** `Tidak Dapat Pertanyaan` berarti
   responden tidak ditanyai karena berada di luar sasaran usia instrumen, bukan berarti
   datanya hilang. Diabetes & Hipertensi hanya mencakup usia 15+, Kanker Paru 45+,
   sedangkan TBC mencakup semua usia. Rentang ini dicantumkan pada setiap grafik.

3. **Prevalensi "berisiko diabetes" bukan temuan epidemiologis yang berdiri sendiri.**
   Label tersebut praktis merupakan aturan `IMT > 25` (lihat `Analisis Data Mentah.ipynb`
   Bagian 8), sehingga sebagian besar hanyalah pernyataan ulang dari prevalensi overweight.

In [ ]:

# ===================== IMPORTS & LOAD =====================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modul bersama + konfigurasi wilayah. Akar proyek dicari dari folder kerja,
# jadi tidak ada satu pun jalur absolut di notebook ini: seluruh folder boleh
# dipindah atau diserahkan ke orang lain tanpa sel ini perlu diubah.
WILAYAH = "Bantul"

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg      # jalur + cara baca berkas mentah tiap wilayah
import bmi_standards as bs      # standar BMI & batas plausibilitas per usia
import viz_style as vz          # gaya grafik seragam untuk semua notebook

vz.apply_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)

DATA_DIR = cfg.folder_data(WILAYAH)

df_raw = cfg.baca_mentah(WILAYAH)
df = pd.read_csv(os.path.join(DATA_DIR, "dataset_master.csv"))

TARGETS = [
    "Berisiko Diabetes Melitus",
    "Berisiko Hipertensi",
    "Berisiko Kanker Paru",
    "Berisiko TBC",
]

# Rentang usia efektif tiap target (hasil gerbang eligibilitas kuesioner).
ELIGIBLE = {
    "Berisiko Diabetes Melitus": "15+",
    "Berisiko Hipertensi": "15+",
    "Berisiko Kanker Paru": "45+",
    "Berisiko TBC": "semua usia",
}

print(f"Data mentah : {df_raw.shape[0]:,} baris")
print(f"Data bersih : {df.shape[0]:,} baris")
display(df.head())


---
# Bagian A — Apa yang Berubah Setelah Pembersihan


## A1. Jejak Pembersihan per Tahap

In [ ]:

# ===================== WATERFALL PEMBERSIHAN =====================
# Dibaca dari file audit yang diekspor notebook Persiapan Data, supaya angkanya
# tidak dihitung ulang (satu sumber kebenaran).
audit = pd.read_csv(os.path.join(DATA_DIR, "audit_pembersihan.csv"))
audit["Selisih"] = audit["Baris tersisa"].diff().fillna(0).astype(int)
display(audit)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
bars = ax.bar(range(len(audit)), audit["Baris tersisa"], color="steelblue")
ax.set_xticks(range(len(audit)))
ax.set_xticklabels([t.split(". ", 1)[-1] for t in audit["Tahap"]],
                   rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Baris tersisa")
ax.set_title("Jumlah Baris di Setiap Tahap Pembersihan")

for i, (v, d) in enumerate(zip(audit["Baris tersisa"], audit["Selisih"])):
    ax.text(i, v, f"{v:,}" + (f"\n({d:+,})" if d else ""),
            ha="center", va="bottom", fontsize=8)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

hilang = df_raw.shape[0] - df.shape[0]
print(f"Total baris dibuang: {hilang:,} ({hilang / df_raw.shape[0] * 100:.2f}% dari data awal)")


Hanya sekitar 1% data yang dibuang. Angka ini penting untuk laporan: pembersihan di sini
**tidak** memangkas data secara agresif — sebagian besar perbaikan dilakukan dengan
membetulkan baris (menukar balik tinggi/berat), bukan membuangnya.


## A2. Dampak Pembersihan pada Statistik Deskriptif

In [ ]:

# ===================== SEBELUM vs SESUDAH =====================
banding = pd.DataFrame({
    "SEBELUM — mean": [df_raw[c].mean() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SEBELUM — std": [df_raw[c].std() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SEBELUM — max": [df_raw[c].max() for c in ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]],
    "SESUDAH — mean": [df[c].mean() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
    "SESUDAH — std": [df[c].std() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
    "SESUDAH — max": [df[c].max() for c in ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]],
}, index=["Usia", "Tinggi Badan", "Berat Badan", "IMT"]).round(2)

display(banding)


### Inilah alasan pembersihan bukan sekadar urusan modeling

Rata-rata IMT mentah adalah **69.5** — angka yang secara fisiologis mustahil sebagai
rata-rata populasi. Setelah dibersihkan, angkanya menjadi **22.3**, yang wajar untuk
populasi Indonesia.

Standar deviasi IMT mentah mencapai ribuan, akibat beberapa baris dengan tinggi badan
salah input. Artinya: **tanpa pembersihan, tidak ada satu pun statistik deskriptif dari
data ini yang bisa dilaporkan** — bukan hanya modelnya yang rusak, melainkan setiap
rata-rata, setiap simpangan baku, dan setiap grafik distribusi.

In [ ]:

# ===================== DISTRIBUSI SEBELUM vs SESUDAH =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

raw_cols = ["Usia", "Tinggibadan", "Berat Badan", "Index Masa Tubuh"]
clean_cols = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]

for i, col in enumerate(raw_cols):
    lo, hi = df_raw[col].quantile([0.01, 0.99])
    sns.histplot(df_raw[col].clip(lo, hi), bins=40, kde=True,
                 ax=axes[0, i], color="lightcoral")
    axes[0, i].set_title(f"{col} — SEBELUM\n(dipotong p1-p99 agar terbaca)", fontsize=10)

for i, col in enumerate(clean_cols):
    sns.histplot(df[col], bins=40, kde=True, ax=axes[1, i], color="mediumseagreen")
    axes[1, i].set_title(f"{col} — SESUDAH", fontsize=10)

plt.tight_layout()
plt.show()


---
# Bagian B — Siapa Populasinya dan Siapa yang Berisiko


## B1. Profil Demografi

In [ ]:

# ===================== KELOMPOK USIA =====================
AGE_BINS = [0, 5, 15, 25, 35, 45, 55, 65, 120]
AGE_LABELS = ["0-4", "5-14", "15-24", "25-34", "35-44", "45-54", "55-64", "65+"]
df["Kelompok_Usia"] = pd.cut(df["Usia"], bins=AGE_BINS, labels=AGE_LABELS, right=False)

ringkas = pd.DataFrame({
    "Jumlah": df["Kelompok_Usia"].value_counts().sort_index(),
    "%": (df["Kelompok_Usia"].value_counts(normalize=True).sort_index() * 100).round(2),
})
display(ringkas)

print(df["Jeniskelamin"].value_counts().to_string())
print()
print(df["segmen"].value_counts().to_string())

In [ ]:

# ===================== PIRAMIDA PENDUDUK & SEGMEN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Piramida penduduk ---
pyr = df.groupby(["Kelompok_Usia", "Jeniskelamin"], observed=True).size().unstack(fill_value=0)
axes[0].barh(pyr.index.astype(str), -pyr.get("L", 0), color="#4A90D9", label="Laki-laki")
axes[0].barh(pyr.index.astype(str), pyr.get("P", 0), color="#E8748C", label="Perempuan")
axes[0].set_title("Piramida Peserta Skrining")
axes[0].set_xlabel("Jumlah catatan skrining")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].legend()
axes[0].xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{abs(int(x)):,}")
)

# --- Segmen ---
seg = df["segmen"].value_counts()
axes[1].barh(seg.index[::-1], seg.values[::-1], color="steelblue")
axes[1].set_title("Distribusi Segmen Kepesertaan")
axes[1].set_xlabel("Jumlah catatan skrining")
for i, v in enumerate(seg.values[::-1]):
    axes[1].text(v, i, f" {v:,}", va="center", fontsize=8)

plt.tight_layout()
plt.show()


`segmen` adalah kategori kepesertaan BPJS, dan merupakan **satu-satunya proksi sosial
ekonomi** yang tersedia di data ini:

| Kode | Arti |
|---|---|
| APBN / APBD | Iuran ditanggung pemerintah pusat / daerah (umumnya kelompok kurang mampu) |
| PPU PN / PPU BU | Pekerja penerima upah — penyelenggara negara / badan usaha |
| PBPU | Pekerja bukan penerima upah (mandiri) |
| BP PN / BP SWASTA | Bukan pekerja (termasuk pensiunan) |

Karena itu, gradien prevalensi antar `segmen` di bagian B3 layak diperhatikan.


## B2. Profil Antropometri & Status Gizi

In [ ]:

# ===================== IMT PER USIA & JENIS KELAMIN =====================
fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

sns.boxplot(data=df, x="Kelompok_Usia", y="IMT_recalculated",
            hue="Jeniskelamin", ax=axes[0], showfliers=False,
            palette={"L": "#4A90D9", "P": "#E8748C"})
axes[0].set_title("Distribusi IMT per Kelompok Usia & Jenis Kelamin")
axes[0].set_ylabel("IMT (kg/m2)")
axes[0].axhline(25, color="darkorange", ls="--", lw=1, label="Ambang overweight (25)")
axes[0].axhline(18.5, color="steelblue", ls="--", lw=1, label="Ambang underweight (18.5)")
axes[0].legend(fontsize=8)

# Prevalensi overweight+obese (dewasa saja -- kategori anak memakai standar berbeda)
dewasa = df[df["Usia"] > bs.CHILD_MAX_AGE]
ow = (dewasa.groupby(["Kelompok_Usia", "Jeniskelamin"], observed=True)["Kategori_BMI_WHO4"]
             .apply(lambda s: s.isin(["Overweight", "Obese"]).mean() * 100)
             .unstack())
ow.plot(kind="bar", ax=axes[1], color={"L": "#4A90D9", "P": "#E8748C"})
axes[1].set_title("Prevalensi Overweight + Obese (dewasa > 20 th)")
axes[1].set_ylabel("% dari kelompok")
axes[1].set_xlabel("Kelompok Usia")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== STATUS GIZI DEWASA =====================
gizi = pd.DataFrame({
    "Jumlah": dewasa["Kategori_BMI_WHO4"].value_counts(),
    "%": (dewasa["Kategori_BMI_WHO4"].value_counts(normalize=True) * 100).round(2),
}).reindex(["Underweight", "Normal", "Overweight", "Obese"])
display(gizi)

print(f"Populasi dewasa (> {bs.CHILD_MAX_AGE} th): {len(dewasa):,} catatan skrining")
print(f"Underweight       : {gizi.loc['Underweight', '%']:.1f}%")
print(f"Overweight + Obese: {gizi.loc[['Overweight', 'Obese'], '%'].sum():.1f}%")


Perhatikan bahwa **kekurangan gizi dan kelebihan gizi hadir bersamaan** dalam populasi yang
sama — pola yang dikenal sebagai *double burden of malnutrition*. Ini temuan kesehatan
masyarakat yang berdiri sendiri, terlepas dari model prediksi apa pun, dan berimplikasi
pada program intervensi yang perlu menyasar dua arah sekaligus.


## B3. Prevalensi Risiko per Subkelompok

In [ ]:

# ===================== PREVALENSI PER TARGET =====================
def prevalensi(col):
    # % Berisiko di antara responden yang benar-benar ditanyai.
    s = df[col]
    ditanya = s.isin(["Berisiko", "Tidak Berisiko"])
    return df[ditanya], (s[ditanya] == "Berisiko")


ringkas = []
for col in TARGETS:
    sub, pos = prevalensi(col)
    ringkas.append({
        "Target": col.replace("Berisiko ", ""),
        "Rentang usia": ELIGIBLE[col],
        "N ditanyai": f"{len(sub):,}",
        "Berisiko": f"{int(pos.sum()):,}",
        "Prevalensi (%)": round(pos.mean() * 100, 2),
    })

display(pd.DataFrame(ringkas))
print("Prevalensi dihitung terhadap responden yang DITANYAI, bukan terhadap seluruh data.")

In [ ]:

# ===================== PREVALENSI per JENIS KELAMIN & USIA =====================
fig, axes = plt.subplots(2, 4, figsize=vz.FIG_2X4)

for j, col in enumerate(TARGETS):
    sub, pos = prevalensi(col)
    sub = sub.assign(_pos=pos.values)

    # --- per jenis kelamin ---
    g = sub.groupby("Jeniskelamin")["_pos"].mean() * 100
    axes[0, j].bar(g.index, g.values, color=["#4A90D9", "#E8748C"])
    axes[0, j].set_title(f"{col.replace('Berisiko ', '')}\n(usia {ELIGIBLE[col]})", fontsize=10)
    axes[0, j].set_ylabel("% Berisiko")
    for i, v in enumerate(g.values):
        axes[0, j].text(i, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
    axes[0, j].margins(y=0.15)

    # --- per kelompok usia ---
    g2 = sub.groupby("Kelompok_Usia", observed=True)["_pos"].mean() * 100
    g2 = g2.dropna()
    axes[1, j].plot(g2.index.astype(str), g2.values, marker="o", color="crimson")
    axes[1, j].set_ylabel("% Berisiko")
    axes[1, j].set_xlabel("Kelompok Usia")
    axes[1, j].tick_params(axis="x", rotation=45)
    axes[1, j].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== PREVALENSI per SEGMEN =====================
seg_prev = {}
for col in TARGETS:
    sub, pos = prevalensi(col)
    seg_prev[col.replace("Berisiko ", "")] = (
        sub.assign(_pos=pos.values).groupby("segmen")["_pos"].mean() * 100
    ).round(2)

seg_df = pd.DataFrame(seg_prev)
display(seg_df)

fig, ax = plt.subplots(figsize=vz.FIG_SINGLE)
seg_df.plot(kind="bar", ax=ax)
ax.set_title("Prevalensi Risiko per Segmen Kepesertaan")
ax.set_ylabel("% Berisiko")
ax.set_xlabel("Segmen")
ax.tick_params(axis="x", rotation=0)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


Gradien antar `segmen` cukup jelas, terutama pada **TBC**. Karena `segmen` mencerminkan
status kepesertaan — dan secara tidak langsung status sosial ekonomi — pola ini konsisten
dengan literatur bahwa risiko TBC berkaitan erat dengan kondisi sosial ekonomi.

**Peringatan dalam membaca angka ini:** `segmen` juga berkorelasi dengan usia (misalnya
segmen "bukan pekerja" banyak berisi pensiunan). Jadi sebagian dari perbedaan ini adalah
efek usia yang menyamar sebagai efek sosial ekonomi. Tabel di bagian B5 memecah keduanya.


## B4. Komorbiditas — Berapa Risiko yang Ditanggung Sekaligus?

Analisis ini **hanya bermakna pada usia 45+**, karena hanya pada kelompok itulah keempat
pertanyaan sama-sama ditanyakan. Di bawah usia 45, Kanker Paru tidak ditanyakan sama
sekali, sehingga "tidak berisiko kanker paru" pada anak muda berarti *tidak diketahui*,
bukan *negatif*.

In [ ]:

# ===================== KOMORBIDITAS (USIA 45+) =====================
komorbid = df[df["Usia"] >= 45].copy()
for col in TARGETS:
    komorbid[col] = komorbid[col].map({"Tidak Berisiko": 0, "Berisiko": 1})
komorbid = komorbid.dropna(subset=TARGETS)

print(f"Populasi usia 45+ dengan keempat pertanyaan terjawab: {len(komorbid):,}")

fig, axes = plt.subplots(1, 2, figsize=vz.FIG_1X2)

# --- Heatmap ko-okurensi ---
short = [c.replace("Berisiko ", "") for c in TARGETS]
co = komorbid[TARGETS].corr()
co.index = co.columns = short
sns.heatmap(co, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, ax=axes[0])
axes[0].set_title("Korelasi Antar Risiko (usia 45+)")

# --- Jumlah risiko yang ditanggung ---
n_risk = komorbid[TARGETS].sum(axis=1).astype(int)
vc = n_risk.value_counts().sort_index()
axes[1].bar(vc.index.astype(str), vc.values, color="steelblue")
axes[1].set_title("Jumlah Risiko yang Ditanggung Bersamaan (usia 45+)")
axes[1].set_xlabel("Banyaknya risiko")
axes[1].set_ylabel("Jumlah catatan skrining")
for i, v in enumerate(vc.values):
    axes[1].text(i, v, f"{v:,}\n({v/len(komorbid)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=8)
axes[1].margins(y=0.18)

plt.tight_layout()
plt.show()

In [ ]:

# ===================== KOMBINASI RISIKO TERBANYAK =====================
kombinasi = (
    komorbid[TARGETS].astype(int)
    .apply(lambda r: " + ".join([s for s, v in zip(short, r) if v]) or "(tidak ada)", axis=1)
    .value_counts()
    .head(10)
)
display(pd.DataFrame({
    "Jumlah": kombinasi,
    "%": (kombinasi / len(komorbid) * 100).round(2),
}))


## B5. Subkelompok dengan Risiko Tertinggi

Tabel ini yang paling langsung bisa dipakai puskesmas: kombinasi kelompok usia, jenis
kelamin, dan segmen mana yang paling perlu diprioritaskan untuk tindak lanjut.

In [ ]:

# ===================== SUBKELOMPOK RISIKO TERTINGGI =====================
MIN_N = 300      # ambang minimal agar persentase tidak menyesatkan

def subkelompok_teratas(col, k=10):
    sub, pos = prevalensi(col)
    sub = sub.assign(_pos=pos.values)
    g = (sub.groupby(["Kelompok_Usia", "Jeniskelamin", "segmen"], observed=True)
            .agg(N=("_pos", "size"), Prevalensi=("_pos", "mean")))
    g = g[g["N"] >= MIN_N].copy()
    g["Prevalensi (%)"] = (g["Prevalensi"] * 100).round(2)
    return g.drop(columns="Prevalensi").sort_values("Prevalensi (%)", ascending=False).head(k)


for col in TARGETS:
    print("=" * 78)
    print(f"{col}   (usia {ELIGIBLE[col]}, minimal N={MIN_N})")
    print("=" * 78)
    display(subkelompok_teratas(col))


## B6. Ringkasan

In [ ]:

# ===================== RINGKASAN AKHIR =====================
print("=" * 74)
print("RINGKASAN — POPULASI SKRINING BANTUL 2025 (data bersih)")
print("=" * 74)
print(f"Catatan skrining      : {len(df):,}")
print(f"Rentang usia          : {int(df['Usia'].min())} - {int(df['Usia'].max())} tahun")
print(f"Komposisi jenis kelamin: "
      f"{(df['Jeniskelamin'] == 'L').mean()*100:.1f}% L / {(df['Jeniskelamin'] == 'P').mean()*100:.1f}% P")
print(f"IMT rata-rata         : {df['IMT_recalculated'].mean():.2f} "
      f"(mentah: {df_raw['Index Masa Tubuh'].mean():.1f})")
print()
print("Prevalensi risiko (di antara yang ditanyai):")
for col in TARGETS:
    sub, pos = prevalensi(col)
    print(f"  {col.replace('Berisiko ', ''):<20} {pos.mean()*100:>6.2f}%   "
          f"(usia {ELIGIBLE[col]}, n={len(sub):,})")


### Temuan utama

1. **Beban gizi ganda.** Underweight dan overweight/obese hadir bersamaan dalam populasi
   yang sama, sehingga intervensi gizi tidak bisa hanya menyasar satu arah.

2. **Gradien sosial ekonomi pada TBC.** Prevalensi risiko TBC berbeda nyata antar segmen
   kepesertaan, konsisten dengan kaitan TBC dan kondisi sosial ekonomi — meskipun sebagian
   efeknya bercampur dengan usia.

3. **Kanker Paru sangat timpang menurut jenis kelamin.** Perbedaan laki-laki vs perempuan
   jauh lebih besar daripada perbedaan antar kelompok usia, konsisten dengan kebiasaan
   merokok sebagai faktor dominan — meskipun status merokok sendiri tidak tersedia di data.

4. **Prevalensi risiko naik seiring usia** pada Diabetes, Hipertensi, dan TBC, tetapi
   relatif datar pada Kanker Paru — karena populasinya sudah terpotong pada usia 45+.

### Keterbatasan

- Semua angka **per catatan skrining**, bukan per orang; `id` teracak sehingga deduplikasi
  individu tidak mungkin.
- Angka prevalensi berlaku untuk **peserta yang ikut skrining**, bukan untuk seluruh
  penduduk Bantul — ada kemungkinan bias partisipasi.
- **Prevalensi "berisiko diabetes" tidak berdiri sendiri**: label tersebut praktis
  merupakan aturan `IMT > 25`, sehingga sebagian besar hanyalah pernyataan ulang dari
  prevalensi overweight.
- Faktor risiko penting tidak tersedia di data: riwayat keluarga, lingkar perut, status
  merokok, dan aktivitas fisik.